In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
from tqdm import tqdm

In [2]:

class ResNeXtBottleneck(nn.Module):
    def __init__(self, in_channels, out_channels, stride, cardinality=32, base_width=4):
        super(ResNeXtBottleneck, self).__init__()
        D = cardinality * base_width

        self.conv1 = nn.Conv2d(in_channels, D , kernel_size=1, bias = False)
        self.bn1 = nn.BatchNorm2d(D)
        self.conv2 = nn.Conv2d(D , D , kernel_size=3, stride=stride, padding=1, groups=cardinality, bias= False)
        self.bn2 = nn.BatchNorm2d(D)
        self.conv3 = nn.Conv2d(D , out_channels , kernel_size=1, bias = False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels))

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out += identity
        out = self.relu(out)
        return out

In [3]:

class ResNeXt(nn.Module):
    def __init__(self, num_blocks, cardinality, bottleneck_width, num_classes=10):
        super(ResNeXt, self).__init__()
        self.in_channels = 64
        self.cardinality = cardinality
        self.base_width = bottleneck_width

        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(256, num_blocks[0] , stride=1)
        self.layer2 = self._make_layer(512, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(1024, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(2048, num_blocks[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1 , 1))
        self.fc = nn.Linear(2048, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(ResNeXtBottleneck(self.in_channels, out_channels, stride, self.cardinality, self.base_width))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(ResNeXtBottleneck(out_channels, out_channels, 1, self.cardinality, self.base_width))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

def resnext50_cifar():
    return ResNeXt(num_blocks=[3, 4, 6, 3], cardinality=32, bottleneck_width=4)


In [4]:
num_epochs = 3
batch_size = 32
learning_rate = 0.01

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [5]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

100%|██████████| 170M/170M [00:03<00:00, 43.7MB/s]


In [7]:
model = resnext50_cifar().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4)

In [10]:
for epoch in range(num_epochs):
    model.train()
    for i, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Accuracy on 10000 test images: {100 * correct / total:.2f}%')

print("Training finished.")

Epoch 1/3: 100%|██████████| 1563/1563 [02:10<00:00, 11.94it/s]


Epoch [1/3], Loss: 1.3507
Accuracy on 10000 test images: 50.93%


Epoch 2/3: 100%|██████████| 1563/1563 [02:14<00:00, 11.63it/s]


Epoch [2/3], Loss: 1.2283
Accuracy on 10000 test images: 69.50%


Epoch 3/3: 100%|██████████| 1563/1563 [02:14<00:00, 11.64it/s]


Epoch [3/3], Loss: 0.7931
Accuracy on 10000 test images: 76.32%
Training finished.
